In [1]:
import pandas as pd
import sys
import numpy as np
sys.path.append('..')
from src.drift import build_reference,compute_psi,drift_evidently

In [2]:
df = pd.read_csv("../data/simulated_data.csv")

In [3]:
print(df[["userDataReadIops","userDataWriteIops","readLatencyMs","writeLatencyMs","cpuPercent","memoryPercent"]].describe())

       userDataReadIops  userDataWriteIops  readLatencyMs  writeLatencyMs  \
count       9940.000000        9940.000000    9940.000000     9940.000000   
mean        4865.447566        2994.942482       2.146910        3.396639   
std         1045.759820         854.441029       1.337121        2.385057   
min            0.043663           0.503411       0.172457        0.767198   
25%         4299.256336        2592.389526       1.657587        2.666533   
50%         4987.877613        3021.137651       2.006273        3.017769   
75%         5664.356392        3410.820741       2.348337        3.376885   
max         6605.032438        8788.281420      17.163125       23.737978   

        cpuPercent  memoryPercent  
count  9940.000000    9940.000000  
mean     39.630412      59.403157  
std       6.276592       7.739071  
min       0.042804       0.004711  
25%      36.539684      56.490809  
50%      39.978739      59.986709  
75%      43.356483      63.412463  
max      57.682658

In [4]:
print(df[df["label"]==0][["userDataReadIops","userDataWriteIops","readLatencyMs","writeLatencyMs","cpuPercent","memoryPercent"]].describe())

       userDataReadIops  userDataWriteIops  readLatencyMs  writeLatencyMs  \
count       9040.000000        9040.000000    9040.000000     9040.000000   
mean        5046.312621        3029.294107       1.993587        2.996580   
std          718.369438         439.314336       0.495692        0.500936   
min         3389.870903        1902.951938       0.172457        0.767198   
25%         4400.362353        2648.479851       1.649592        2.653631   
50%         5081.457442        3052.034316       1.996727        2.996550   
75%         5692.752940        3408.763113       2.329867        3.337696   
max         6605.032438        4015.424573       3.845812        4.863917   

        cpuPercent  memoryPercent  
count  9040.000000    9040.000000  
mean     40.020996      60.010417  
std       4.997287       5.030015  
min      18.523045      39.211331  
25%      36.663151      56.620020  
50%      40.033069      60.075310  
75%      43.389407      63.446591  
max      57.682658

In [3]:
ref = build_reference(df, ["userDataReadIops","userDataWriteIops","readLatencyMs","writeLatencyMs","cpuPercent","memoryPercent"])
print(ref["readLatencyMs"]["expected"])

[0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1]


In [4]:
drifted = df.copy()
drifted["readLatencyMs"] = drifted["readLatencyMs"] + 10
print(compute_psi(drifted, ref, ["userDataReadIops","userDataWriteIops","readLatencyMs","writeLatencyMs","cpuPercent","memoryPercent"]))

{'userDataReadIops': np.float64(0.0), 'userDataWriteIops': np.float64(0.0), 'readLatencyMs': np.float64(8.289396330278864), 'writeLatencyMs': np.float64(0.0), 'cpuPercent': np.float64(0.0), 'memoryPercent': np.float64(0.0)}


In [6]:
drift_evident = drift_evidently(df,drifted,["userDataReadIops","userDataWriteIops","readLatencyMs","writeLatencyMs","cpuPercent","memoryPercent"])
print(drift_evident)

{'metrics': [{'id': '15e89f895b482f9b84ba7274ed18a106', 'metric_name': 'DriftedColumnsCount(drift_share=0.5)', 'config': {'type': 'evidently:metric_v2:DriftedColumnsCount', 'drift_share': 0.5}, 'value': {'count': 1.0, 'share': 0.16666666666666666}}, {'id': 'bcf6007c521d8e98bf5a5e6e23a0bfae', 'metric_name': 'ValueDrift(column=userDataReadIops,method=Wasserstein distance (normed),threshold=0.1)', 'config': {'type': 'evidently:metric_v2:ValueDrift', 'column': 'userDataReadIops', 'method': 'Wasserstein distance (normed)', 'threshold': 0.1}, 'value': np.float64(0.0)}, {'id': '7edc53ee0434da3d2ae409a0ab8a07c1', 'metric_name': 'ValueDrift(column=userDataWriteIops,method=Wasserstein distance (normed),threshold=0.1)', 'config': {'type': 'evidently:metric_v2:ValueDrift', 'column': 'userDataWriteIops', 'method': 'Wasserstein distance (normed)', 'threshold': 0.1}, 'value': np.float64(0.0)}, {'id': '0e908e40b7575486294ab1e039683568', 'metric_name': 'ValueDrift(column=readLatencyMs,method=Wasserstei

In [7]:
drift_scores ={}
for feature in drift_evident["metrics"]:
    if "ValueDrift" in feature["metric_name"]:
        drift_scores[feature["config"]["column"]] = feature["value"]

print(drift_scores)


{'userDataReadIops': np.float64(0.0), 'userDataWriteIops': np.float64(0.0), 'readLatencyMs': np.float64(7.479132764345477), 'writeLatencyMs': np.float64(0.0), 'cpuPercent': np.float64(0.0), 'memoryPercent': np.float64(0.0)}
